# Classification Evaluation Metrics: Confusion Matrix, Imbalance, and Multi-Class Scoring

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Classification involves assigning discrete category labels to observations. In real-world data science, relying solely on **Accuracy** can be catastrophically misleading when classes are imbalanced. This notebook builds complete diagnostic pipelines from first principles: the Confusion Matrix, Precision, Recall, Specificity, F1-Score, multi-class averaging (Macro, Micro, Weighted), and Log Loss.

## Learning Outcomes

- Understand the Accuracy Paradox in imbalanced datasets.
- Construct multi-class confusion matrices from scratch using NumPy.
- Compute Precision, Recall, Specificity, and F1-score from confusion matrix components.
- Aggregate multi-class performance using Macro, Micro, and Weighted averaging schemes.
- Compute categorical Log Loss (Cross-Entropy) on probabilistic predictions.
- Evaluate multi-class classification predictions on `data/classification_eval_data.csv`.


In [ ]:
import numpy as np

# Demonstrating the Accuracy Paradox
# Scenario: Medical screening where 98% of subjects are healthy (0) and 2% have disease (1)
true_disease = np.zeros(100, dtype=int)
true_disease[:2] = 1  # 2 positive cases

# Trivial majority predictor that always predicts healthy (0)
trivial_pred = np.zeros(100, dtype=int)

accuracy = np.mean(true_disease == trivial_pred)
print(f"Trivial predictor accuracy: {accuracy:.2%}")
print("Disease detected: 0/2 (Model is useless despite 98% accuracy!)")

## 1. The Confusion Matrix from First Principles

In a confusion matrix $C$, the element $C_{i, j}$ represents the count of samples whose true class is $i$ and whose predicted class is $j$:
$$\text{Rows} \to \text{True Classes}, \quad \text{Columns} \to \text{Predicted Classes}$$


In [ ]:
import numpy as np


def compute_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int | None = None,
) -> np.ndarray:
    """Compute confusion matrix for arbitrary integer labels [0, n_classes - 1]."""
    assert y_true.shape == y_pred.shape, "Shape mismatch"
    if n_classes is None:
        n_classes = max(int(np.max(y_true)), int(np.max(y_pred))) + 1

    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred, strict=True):
        cm[t, p] += 1
    return cm


# Test on toy 3-class data
t_sample = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
p_sample = np.array([0, 0, 1, 1, 1, 2, 2, 0, 2])

cm = compute_confusion_matrix(t_sample, p_sample, n_classes=3)
print("Confusion Matrix (Rows=True, Cols=Pred):")
print(cm)

## 2. Binary Metrics: Precision, Recall, Specificity, and F1-Score

For binary classification (Positive $= 1$, Negative $= 0$):
- $\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$ (How many declared positives were actually positive?)
- $\text{Recall (Sensitivity)} = \frac{\text{TP}}{\text{TP} + \text{FN}}$ (How many actual positives were successfully found?)
- $\text{Specificity} = \frac{\text{TN}}{\text{TN} + \text{FP}}$ (True negative rate)
- $\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ (Harmonic mean of precision and recall)


In [ ]:
import numpy as np


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    cm = compute_confusion_matrix(y_true, y_pred, n_classes=2)
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    acc = (tp + tn) / len(y_true)

    return {
        "Accuracy": float(acc),
        "Precision": float(precision),
        "Recall": float(recall),
        "Specificity": float(specificity),
        "F1-Score": float(f1),
    }


# Evaluate on unbalanced sample
y_t = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_p = np.array([1, 1, 1, 0, 0, 0, 0, 1, 0, 0])

metrics = binary_metrics(y_t, y_p)
for k, v in metrics.items():
    print(f"{k:<12}: {v:.4f}")

## 3. Multi-Class Averaging: Macro vs Micro vs Weighted

In multi-class settings ($K > 2$), we treat each class $k$ as a one-vs-rest binary problem and aggregate:
- **Macro-average:** Unweighted arithmetic mean across all per-class metrics (gives equal importance to every class regardless of frequency).
- **Micro-average:** Sum global TP, FP, FN across all classes before calculating metric (dominated by majority classes).
- **Weighted-average:** Weighted average of per-class metrics by true class support $N_k$.


In [ ]:
from typing import Any

import numpy as np


def multiclass_classification_report(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
) -> dict[str, Any]:
    n_classes = len(class_names)
    cm = compute_confusion_matrix(y_true, y_pred, n_classes=n_classes)

    precisions = []
    recalls = []
    f1s = []
    supports = []

    for k in range(n_classes):
        tp = cm[k, k]
        fp = np.sum(cm[:, k]) - tp
        fn = np.sum(cm[k, :]) - tp
        support = np.sum(cm[k, :])

        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

        precisions.append(p)
        recalls.append(r)
        f1s.append(f)
        supports.append(support)

    total_samples = len(y_true)
    accuracy = float(np.trace(cm) / total_samples)

    macro_p = float(np.mean(precisions))
    macro_r = float(np.mean(recalls))
    macro_f = float(np.mean(f1s))

    weights = np.array(supports) / total_samples
    weighted_p = float(np.sum(np.array(precisions) * weights))
    weighted_r = float(np.sum(np.array(recalls) * weights))
    weighted_f = float(np.sum(np.array(f1s) * weights))

    return {
        "confusion_matrix": cm,
        "classes": {
            class_names[k]: {
                "precision": precisions[k],
                "recall": recalls[k],
                "f1": f1s[k],
                "support": int(supports[k]),
            }
            for k in range(n_classes)
        },
        "accuracy": accuracy,
        "macro_avg": {"precision": macro_p, "recall": macro_r, "f1": macro_f},
        "weighted_avg": {"precision": weighted_p, "recall": weighted_r, "f1": weighted_f},
    }

## 4. End-to-End Evaluation on `data/classification_eval_data.csv`


In [ ]:
import csv
from pathlib import Path

import numpy as np

csv_path = Path("../data/classification_eval_data.csv")
y_true_all = []
y_pred_all = []
prob_matrix = []

with csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        y_true_all.append(int(row["true_class_id"]))
        y_pred_all.append(int(row["pred_class_id"]))
        prob_matrix.append(
            [float(row["prob_cat"]), float(row["prob_dog"]), float(row["prob_bird"])]
        )

y_true_arr = np.array(y_true_all)
y_pred_arr = np.array(y_pred_all)
probs_arr = np.array(prob_matrix)

class_names = ["cat", "dog", "bird"]
report = multiclass_classification_report(y_true_arr, y_pred_arr, class_names)

print(f"{'Class':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Support':<8}")
print("-" * 56)
for cname, stats in report["classes"].items():
    print(
        f"{cname:<10} | {stats['precision']:<10.4f} | {stats['recall']:<10.4f} | {stats['f1']:<10.4f} | {stats['support']:<8}"
    )
print("-" * 56)
print(f"Overall Accuracy:  {report['accuracy']:.4f}")
print(f"Macro F1-Score:    {report['macro_avg']['f1']:.4f}")
print(f"Weighted F1-Score: {report['weighted_avg']['f1']:.4f}")

# Compute Categorical Log Loss
eps = 1e-15
clipped_p = np.clip(probs_arr, eps, 1.0 - eps)
one_hot = np.zeros_like(clipped_p)
for i, t in enumerate(y_true_arr):
    one_hot[i, t] = 1.0
log_loss = -float(np.mean(np.sum(one_hot * np.log(clipped_p), axis=1)))
print(f"Categorical Log Loss: {log_loss:.4f}")